In [58]:
%pip install -q pdfminer.six pdfplumber pymupdf pytesseract pdf2image pillow \
requests beautifulsoup4 lxml tqdm pandas numpy

In [59]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import re
import time
import shutil
import logging
import threading
import urllib.request

from datetime import date
from io import BytesIO
from glob import glob

import requests
import pandas as pd
import pdfplumber
import fitz
import pytesseract

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, wait
from pdfminer.high_level import extract_text
from pdf2image import convert_from_path
from PIL import Image
from tqdm import tqdm

In [60]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
# =========================================================
# DIRECTORY CONFIGURATION
# =========================================================

ROOT_DIR = "/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan"

PATH_PDF = os.path.join(
    ROOT_DIR,
    "data/raw_pdf"
)

PATH_OUTPUT = os.path.join(
    ROOT_DIR,
    "data/raw_text"
)

LOG_DIR = os.path.join(
    ROOT_DIR,
    "logs"
)

LOG_PATH = os.path.join(
    LOG_DIR,
    "cleaning.log"
)

MAX_PATH_LENGTH = 260

os.makedirs(PATH_PDF, exist_ok=True)
os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("ROOT_DIR :", ROOT_DIR)
print("PDF_DIR  :", PATH_PDF)
print("TEXT_DIR :", PATH_OUTPUT)
print("LOG_DIR  :", LOG_DIR)

ROOT_DIR : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan
PDF_DIR  : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/raw_pdf
TEXT_DIR : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/raw_text
LOG_DIR  : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/logs


In [62]:
# =========================================================
# LOGGING
# =========================================================

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(
            LOG_PATH,
            mode='w',
            encoding='utf-8'
        )
    ],
    force=True
)

logging.info("===== NOTEBOOK 01 STARTED =====")
print("Logging aktif.")

Logging aktif.


In [63]:
# =========================================================
# THREAD CONFIG
# =========================================================

processed_files = 0
file_lock = threading.Lock()

In [64]:
# =========================================================
#  VALIDATE PATH
# =========================================================

def validate_path(path):

    if len(path) > MAX_PATH_LENGTH:

        raise ValueError(
            f"Path terlalu panjang: {path}"
        )

    return path


for path in [PATH_PDF, PATH_OUTPUT, LOG_DIR]:

    validate_path(path)

print("Validasi path selesai.")

Validasi path selesai.


In [65]:
# =========================================================
# NATURAL SORT
# =========================================================

def natural_sort_key(path):

    filename = os.path.basename(path)

    return [
        int(text) if text.isdigit()
        else text.lower()
        for text in re.split(r'(\d+)', filename)
    ]

In [66]:
# =========================================================
# OPEN PAGE
# =========================================================

def open_page(link):

    count = 0

    while count < 3:

        try:

            response = requests.get(link)

            response.raise_for_status()

            return BeautifulSoup(
                response.text,
                "lxml"
            )

        except Exception as e:

            count += 1

            logging.warning(
                f"Gagal membuka {link} | percobaan={count}"
            )

            time.sleep(5)

    logging.error(f"Gagal total membuka: {link}")

    return None

In [67]:
# =========================================================
# GET PDF URL
# =========================================================

def get_pdf_url(soup):

    try:

        pdf_link = soup.find(
            "a",
            href=re.compile(r"/pdf/")
        )["href"]

        if not pdf_link.startswith("http"):

            pdf_link = (
                "https://putusan3.mahkamahagung.go.id"
                + pdf_link
            )

        return pdf_link

    except Exception as e:

        logging.error(
            f"Gagal mendapatkan PDF URL: {e}"
        )

        return None

In [68]:
# =========================================================
# CHECK DUPLICATE PDF
# =========================================================

def is_url_already_processed(url, path_pdf):

    processed_files_list = [

        f for f in os.listdir(path_pdf)

        if f.endswith('.pdf')
    ]

    return any(

        url.split('/')[-1] in f

        for f in processed_files_list
    )

In [69]:
# =========================================================
# DOWNLOAD PDF
# =========================================================

def download_pdf(url, path_pdf, keyword_url):

    try:

        file = urllib.request.urlopen(url)

        file_name = (
            file.info().get_filename()
            or url.split('/')[-1]
        )

        keyword = "perikanan"

        file_name = re.sub(
            r'[^\w\-_\.]',
            '_',
            file_name.replace(".pdf", "")
        )

        max_base_length = (
            MAX_PATH_LENGTH
            - len(path_pdf)
            - len(f"_{keyword}.pdf")
            - 10
        )

        file_name = file_name[:max_base_length]

        file_name = f"{file_name}_{keyword}.pdf"

        save_path = os.path.join(
            path_pdf,
            file_name
        )

        if len(save_path) > MAX_PATH_LENGTH:

            file_name = (
                f"putusan_{int(time.time())}_{keyword}.pdf"
            )

            save_path = os.path.join(
                path_pdf,
                file_name
            )

        file_content = file.read()

        with open(save_path, "wb") as out_file:

            out_file.write(file_content)

        logging.info(
            f"Berhasil download: {file_name}"
        )

        return file_name

    except Exception as e:

        logging.error(
            f"Gagal download PDF: {e}"
        )

        return None

In [70]:
# =========================================================
# EXTRACT DATA
# =========================================================

def extract_data(link, keyword_url, max_files):

    global processed_files

    with file_lock:

        if processed_files >= max_files:

            logging.info(
                "Batas file tercapai."
            )

            return False

    if is_url_already_processed(link, PATH_PDF):

        logging.info(
            f"Duplicate dilewati: {link}"
        )

        return True

    soup = open_page(link)

    if not soup:

        return True

    link_pdf = get_pdf_url(soup)

    if not link_pdf:

        return True

    file_name = download_pdf(
        link_pdf,
        PATH_PDF,
        keyword_url
    )

    if file_name:

        with file_lock:

            processed_files += 1

            logging.info(
                f"Processed {processed_files}/{max_files}"
            )

            if processed_files >= max_files:

                return False

    return True

In [71]:
# =========================================================
# RUN PROCESS
# =========================================================

def run_process(keyword_url, page, sort_page, max_files):

    global processed_files

    with file_lock:

        if processed_files >= max_files:

            return False

    if keyword_url.startswith("https"):

        link = f"{keyword_url}&page={page}"

    else:

        link = (
            f"https://putusan3.mahkamahagung.go.id/"
            f"search.html?q={keyword_url}&page={page}"
        )

    if sort_page:

        link = (
            f"{link}&obf=TANGGAL_PUTUS&obm=desc"
        )

    logging.info(f"Processing page {page}")

    soup = open_page(link)

    if not soup:

        return False

    links = soup.find_all(
        "a",
        {"href": re.compile("/direktori/putusan")}
    )

    for link in links:

        with file_lock:

            if processed_files >= max_files:

                return False

        full_link = link["href"]

        if not full_link.startswith("http"):

            full_link = (
                "https://putusan3.mahkamahagung.go.id"
                + full_link
            )

        continue_processing = extract_data(
            full_link,
            keyword_url,
            max_files
        )

        if not continue_processing:

            return False

    return True

In [72]:
# =========================================================
# RUN SCRAPER
# =========================================================

def run_scraper(url=None, max_files=30):

    global processed_files

    if not url or not url.startswith("https://"):

        logging.error("URL tidak valid")

        return

    soup = open_page(url)

    if not soup:

        return

    try:

        last_page = int(

            soup.find_all(
                "a",
                {"class": "page-link"}
            )[-1].get(
                "data-ci-pagination-page"
            )
        )

    except:

        last_page = 1

    logging.info(
        f"Total page: {last_page}"
    )

    with ThreadPoolExecutor(max_workers=1) as executor:

        futures = []

        for page in range(1, last_page + 1):

            with file_lock:

                if processed_files >= max_files:

                    break

            future = executor.submit(

                run_process,
                url,
                page,
                True,
                max_files
            )

            futures.append(future)

        wait(futures)

    logging.info(
        f"Download selesai. Total PDF={processed_files}"
    )

    print(
        f"Download selesai. Total PDF={processed_files}"
    )

In [73]:
# =========================================================
# PDF EXTRACTION (PDFMINER)
# =========================================================

def extract_text_pdfminer(pdf_path):

    try:

        text = extract_text(pdf_path)

        if text and len(text.strip()) > 100:

            logging.info(
                f"pdfminer berhasil: {pdf_path}"
            )

            return text

    except Exception as e:

        logging.warning(
            f"pdfminer gagal: {e}"
        )

    return ""

In [74]:
# =========================================================
# PDF EXTRACTION (PDFPLUMBER)
# =========================================================

def extract_text_pdfplumber(pdf_path):

    try:

        all_text = []

        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:

                    all_text.append(page_text)

        text = "\n".join(all_text)

        if len(text.strip()) > 100:

            logging.info(
                f"pdfplumber berhasil: {pdf_path}"
            )

            return text

    except Exception as e:

        logging.warning(
            f"pdfplumber gagal: {e}"
        )

    return ""

In [75]:
# =========================================================
# OCR EXTRACTION
# =========================================================

def extract_text_ocr(pdf_path):

    try:

        images = convert_from_path(pdf_path)

        all_text = []

        for img in images:

            text = pytesseract.image_to_string(
                img,
                lang='ind'
            )

            all_text.append(text)

        text = "\n".join(all_text)

        if len(text.strip()) > 50:

            logging.info(
                f"OCR berhasil: {pdf_path}"
            )

            return text

    except Exception as e:

        logging.warning(
            f"OCR gagal: {e}"
        )

    return ""

In [76]:
# =========================================================
# HYBRID EXTRACTION PIPELINE
# =========================================================

def extract_pdf_text(pdf_path):

    text = extract_text_pdfminer(pdf_path)

    if len(text.strip()) < 100:

        logging.info(
            "Fallback ke pdfplumber"
        )

        text = extract_text_pdfplumber(pdf_path)

    if len(text.strip()) < 100:

        logging.info(
            "Fallback ke OCR"
        )

        text = extract_text_ocr(pdf_path)

    return text

In [77]:
import re
# =========================================================
# CLEAN TEXT
# =========================================================

def clean_text(text):

    if not text:

        return ""

    patterns = [

        r'Direktori Putusan.*?RI',

        r'putusan\.mahkamahagung\.go\.id',

        r'Mahkamah Agung Republik Indonesia',

        r'Disclaimer',

        r'Kepaniteraan.*?RI',

        r'Email\s*:.*',

        r'Telp\s*:.*',

        r'Hal\.\s*\d+',

        r'Dalam hal anda menemukan inakurasi informasi.*?hubungi melalui',

        r'waktu kewaktu.*?hubungi melalui',

        r'halaman\s+\d+.*?putusan',

        r'www\.[^\s]+',

        r'putusan nomor.*?halaman',

    ]

    for pattern in patterns:

        text = re.sub(
            pattern,
            ' ',
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [78]:
# =========================================================
# SAVE TEXT FILE
# =========================================================

def save_text_file(text, index, path):

    if not text:

        return None

    file_name = f"case_{index:03d}.txt"

    save_path = os.path.join(
        path,
        file_name
    )

    with open(
        save_path,
        'w',
        encoding='utf-8'
    ) as f:

        f.write(text)

    logging.info(
        f"Saved: {file_name}"
    )

    return file_name

In [79]:
# =========================================================
# PROCESS PDFS
# =========================================================

def process_pdfs(max_files=30):

    logging.info("PROCESS PDF START")

    # hapus TXT lama
    for file in os.listdir(PATH_OUTPUT):

        file_path = os.path.join(
            PATH_OUTPUT,
            file
        )

        if os.path.isfile(file_path):

            os.remove(file_path)

    # ambil semua PDF
    pdf_files_list = glob(
        os.path.join(PATH_PDF, "*.pdf")
    )

    # urutkan berdasarkan nomor file
    pdf_files_list = sorted(
        pdf_files_list,
        key=natural_sort_key
    )

    print(
        f"Total PDF ditemukan: {len(pdf_files_list)}"
    )

    print("\nUrutan PDF yang diproses:")

    for i, pdf in enumerate(pdf_files_list, start=1):

        print(
            f"{i:02d}. {os.path.basename(pdf)}"
        )

    processed_files_text = 0
    failed_files = []

    for pdf_file in tqdm(pdf_files_list):

        if processed_files_text >= max_files:

            break

        logging.info(
            f"Processing: {pdf_file}"
        )

        text = extract_pdf_text(pdf_file)

        cleaned_text = clean_text(text)

        if cleaned_text:

            processed_files_text += 1

            save_text_file(
                cleaned_text,
                processed_files_text,
                PATH_OUTPUT
            )

        else:

            failed_files.append(
                os.path.basename(pdf_file)
            )

            logging.warning(
                f"Gagal ekstrak: {pdf_file}"
            )

    print("\nHASIL EKSTRAKSI")

    print(
        f"Berhasil : {processed_files_text}"
    )

    print(
        f"Gagal    : {len(failed_files)}"
    )

    if failed_files:

        print("\nDaftar PDF gagal:")

        for f in failed_files:

            print("-", f)

    logging.info(
        f"PROCESS COMPLETE: {processed_files_text}"
    )

    print(
        f"\nProcessing selesai. Total TXT={processed_files_text}"
    )

In [80]:
# =========================================================
# MAIN FUNCTION
# =========================================================
def main():

    print("MASUK MAIN")
    logging.info("MASUK MAIN")

    scraper_url = (
        "https://putusan3.mahkamahagung.go.id/"
        "search.html?q=&jenis_doc=putusan"
        "&cat=8b0e5f0d2f1a0e6a4f3c6b8d1c2e9f11"
        "&jd=&tp=&court=&t_put=&t_reg=&t_upl=&t_pr="
    )

    print("SEBELUM SCRAPER")
    logging.info("SEBELUM SCRAPER")

    run_scraper(
        url=scraper_url,
        max_files=30
    )

    print("SETELAH SCRAPER")
    logging.info("SETELAH SCRAPER")

    process_pdfs(
        max_files=30
    )

    print("SETELAH PROCESS PDF")
    logging.info("SETELAH PROCESS PDF")

In [81]:
main()
logging.shutdown()

print("SEMUA PROSES SELESAI")

MASUK MAIN
SEBELUM SCRAPER
SETELAH SCRAPER
Total PDF ditemukan: 30

Urutan PDF yang diproses:
01. putusan_1.pdf
02. putusan_2.pdf
03. putusan_3.pdf
04. putusan_4.pdf
05. putusan_5.pdf
06. putusan_6.pdf
07. putusan_7.pdf
08. putusan_8.pdf
09. putusan_9.pdf
10. putusan_10.pdf
11. putusan_11.pdf
12. putusan_12.pdf
13. putusan_13.pdf
14. putusan_14.pdf
15. putusan_15.pdf
16. putusan_16.pdf
17. putusan_17.pdf
18. putusan_18.pdf
19. putusan_19.pdf
20. putusan_20.pdf
21. putusan_21.pdf
22. putusan_22.pdf
23. putusan_23.pdf
24. putusan_24.pdf
25. putusan_25.pdf
26. putusan_26.pdf
27. putusan_27.pdf
28. putusan_28.pdf
29. putusan_29.pdf
30. putusan_30.pdf


100%|██████████| 30/30 [00:48<00:00,  1.63s/it]



HASIL EKSTRAKSI
Berhasil : 30
Gagal    : 0

Processing selesai. Total TXT=30
SETELAH PROCESS PDF
SEMUA PROSES SELESAI


In [82]:
# =========================================================
# CHECK RESULT
# =========================================================

txt_files = glob(
    os.path.join(PATH_OUTPUT, "*.txt")
)

txt_files = sorted(
    txt_files,
    key=natural_sort_key
)

print(f"Total TXT: {len(txt_files)}")

for file in txt_files[:3]:

    print("="*80)

    print(os.path.basename(file))

    with open(file, "r", encoding="utf-8") as f:

        text = f.read()

    print(text[:1000])
    print("\n")

Total TXT: 30
case_001.txt
Direktori Putusan Kepaniteraan berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu. Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi melalui : Halaman 1 dari13hal. Put. Nomor :497K/Pid.Sus/2016P U T U S A NNomor :497K/PID.SUS/2016DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAM A H K A M A H A G U N Gmemeriksa perkarapidana khususdalam tingkat kasasi telah memutuskansebagai berikut dalam perkaraParaTerdakwa:I.Nama lengkap:SOPIAN BinRABBIL;Tempat lahir:Toli-toli;Umur/tanggal lahir:36tahun/17 Juni 1979;Jenis Kelamin:Laki-l